# IC50, EC50, and logIC50 models

This notebook demonstrates the logistic model family: `ic50`, `ec50`, and `logic50`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bindcurve as bc

rng = np.random.default_rng(123)
concentrations = np.logspace(-2, 2, 16)

## IC50 inhibition model

In [ ]:
def ic50_curve(x, ymin=0.0, ymax=100.0, IC50=2.0, hill_slope=-1.2):
    return ymin + (ymax - ymin) / (1.0 + (IC50 / x) ** hill_slope)

rows = []
for concentration in concentrations:
    for replicate_id in range(1, 4):
        rows.append({
            'compound_id': 'inhibitor_a',
            'experiment_id': 'exp1',
            'concentration': concentration,
            'replicate_id': f'rep{replicate_id}',
            'response': ic50_curve(concentration) + rng.normal(0, 1.5),
        })

ic50_data = bc.DoseResponseData.from_dataframe(pd.DataFrame(rows), concentration_unit='uM', response_unit='percent')
ic50_results = bc.fit(ic50_data, model='ic50', fixed={'ymin': 0.0, 'ymax': 100.0})
ic50_results.fits_to_dataframe()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bc.plot_curves(ic50_data, ic50_results, ax=ax, confidence_band=True)
ax.legend()
plt.show()

## EC50 activation model

The `ec50` model uses the same 4PL shape but names the midpoint parameter `EC50`.

In [ ]:
def ec50_curve(x, ymin=5.0, ymax=95.0, EC50=3.0, hill_slope=1.3):
    return ymin + (ymax - ymin) / (1.0 + (EC50 / x) ** hill_slope)

rows = []
for concentration in concentrations:
    for replicate_id in range(1, 4):
        rows.append({
            'compound_id': 'activator_a',
            'experiment_id': 'exp1',
            'concentration': concentration,
            'replicate_id': f'rep{replicate_id}',
            'response': ec50_curve(concentration) + rng.normal(0, 1.5),
        })

ec50_data = bc.DoseResponseData.from_dataframe(pd.DataFrame(rows), concentration_unit='uM', response_unit='percent')
ec50_results = bc.fit(ec50_data, model='ec50', fixed={'ymin': 5.0, 'ymax': 95.0})
ec50_results.fits_to_dataframe()

## logIC50 model

The `logic50` model fits on `log10(concentration)` and reports `logIC50`.

In [ ]:
def logic50_curve(concentration, ymin=0.0, ymax=100.0, logIC50=0.25, hill_slope=-1.1):
    log_concentration = np.log10(concentration)
    return ymin + (ymax - ymin) / (1.0 + 10 ** ((logIC50 - log_concentration) * hill_slope))

rows = []
for concentration in concentrations:
    for replicate_id in range(1, 4):
        rows.append({
            'compound_id': 'inhibitor_b',
            'experiment_id': 'exp1',
            'concentration': concentration,
            'replicate_id': f'rep{replicate_id}',
            'response': logic50_curve(concentration) + rng.normal(0, 1.5),
        })

logic50_data = bc.DoseResponseData.from_dataframe(pd.DataFrame(rows), concentration_unit='uM', response_unit='percent')
logic50_results = bc.fit(logic50_data, model='logic50', fixed={'ymin': 0.0, 'ymax': 100.0})
logic50_results.fits_to_dataframe()